In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
%sql
ALTER TABLE dbr_dev_trial.artemzharkov10_gold.knowledge_base_chunks 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
    VectorIndexType
)
import time

In [0]:


w = WorkspaceClient()

source_table = "dbr_dev_trial.artemzharkov10_gold.knowledge_base_chunks"
index_name = "dbr_dev_trial.artemzharkov10_gold.knowledge_base_index"

endpoints = w.vector_search_endpoints.list_endpoints()
existing_endpoints = [e.name for e in endpoints] if endpoints else []

if not existing_endpoints:
    raise Exception("No vector search endpoints found")

endpoint_name = existing_endpoints[0]

while True:
    status = w.vector_search_endpoints.get_endpoint(endpoint_name).endpoint_status.state.value
    if status == "ONLINE":
        break
    time.sleep(30)

indexes = w.vector_search_indexes.list_indexes(endpoint_name)
existing_indexes = [idx.name for idx in indexes] if indexes else []

if index_name in existing_indexes:
    try:
        w.vector_search_indexes.sync_index(index_name=index_name)
        print("Sync started.")
    except Exception as e:
        if "RUNNING" in str(e):
            print("Index is already syncing, skipping.")
        else:
            raise e
else:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="text_content",
                    embedding_model_endpoint_name="databricks-bge-large-en"
                )
            ]
        )
    )